# Analysis of the raw data quality

Based on quality flags generated using `7.4_gauge_raw_data_quality_checks.ipynb`.

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from blue_heart_data_handling import QCFlags, get_gauge_data

In [ ]:
# General settings

# Whether to save plots
save_flag = True

# Dataset path
path_dataset = '..'

# Gauge index
path_index_gauge_data = f'{path_dataset}/index_gauge_data.csv'
df_index = pd.read_csv(path_index_gauge_data)

# Folder to save plots in
path_output_folder = 'outputs/5_raw_data_quality_analysis'
if not os.path.isdir(path_output_folder):
    os.makedirs(path_output_folder)

# Plot settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10
})

In [ ]:
# Load quality flags/labels for each gauge

# Initialise storage of quality summaries for each gauge (proportion and count of data points with each quality flag)
gauge_quality_proportion_summaries = {}
gauge_quality_count_summaries = {}
for gauge_type in df_index['gaugeType'].unique():
    gauge_quality_proportion_summaries[gauge_type] = {}
    gauge_quality_count_summaries[gauge_type] = {}

for index, row in df_index.iterrows():
    location = row['locationName']
    print(f'Loading quality codes for {location}')
    gauge_type = row['gaugeType']
    data_quality_flags_path = row['dataQualityFlagsPath']
    df_gauge_quality = pd.read_csv(f'{path_dataset}/{data_quality_flags_path}', dtype=str)

    # Calculate and store stats for each quality check and overall quality flags for each timeseries
    gauge_quality_proportion_summaries[gauge_type][location] = {}
    gauge_quality_count_summaries[gauge_type][location] = {}
    for column in df_gauge_quality:
        if column != 'timestamp':
            quality_codes_proportions = df_gauge_quality[column].value_counts(normalize=True)
            gauge_quality_proportion_summaries[gauge_type][location][column] = quality_codes_proportions

            quality_codes_counts = df_gauge_quality[column].value_counts()
            gauge_quality_count_summaries[gauge_type][location][column] = quality_codes_counts


## Summarise proportion of each dataset for each gauge categorised with each quality code

In [ ]:
# Summarise proportion of each dataset for each gauge categorised with each quality code

def build_summary_dataframe(gauge_quality_summaries: dict):

    # Initialise list of dataframes
    dfs = []
    for gauge_type in gauge_quality_summaries.keys():
        for location, qc_dict in gauge_quality_summaries[gauge_type].items():
            # Convert each inner series into a DataFrame with a MultiIndex column
            df = pd.concat(
                {qc_location: s for qc_location, s in qc_dict.items()},
                axis=1
            )
            df.index.name = 'flag'
            df['location'] = location
            df['gauge_type'] = gauge_type
            dfs.append(df.reset_index())

    # Combine all locations
    out = pd.concat(dfs, ignore_index=True)
        
    # Pivot to get the desired MultiIndex columns
    df_summary = out.pivot_table(
        index='location',
        columns=['flag', 'gauge_type'],
    )

    # Sort
    df_summary = df_summary.sort_index(axis=1, level=[2, 0, 1])

    return df_summary


df_summary_proportions = build_summary_dataframe(gauge_quality_proportion_summaries)
df_summary_counts = build_summary_dataframe(gauge_quality_count_summaries)

In [ ]:
# Rename summary dataframe columns based on gauge type and parameter, to enable plotting by gauge type for water levels and depths

# Drop Signal-to-noise columns
mask = ~df_summary_proportions.columns.get_level_values(0).str.contains('Signal-to-noise ratio', na=False)
df_summary_proportions = df_summary_proportions.loc[:, mask]

# Build new column names
new_level0_columns = []
for parameter, quality_code, gauge_type in df_summary_proportions.columns:
    
    if 'Water Temperature' in parameter:
        new_name = 'Water Temperature'
        
    elif 'Air Temperature' in parameter:
        new_name = 'Air Temperature'

    elif 'Water' in parameter:
        new_name = f'{gauge_type} water level or depth'
        
    elif 'Level' in parameter:
        new_name = f'{gauge_type} water level or depth'
        
    elif 'Rain' in parameter:
        new_name = 'Rainfall depth'
        
    else:
        new_name = f'{gauge_type} {parameter}'
    
    new_level0_columns.append(new_name)

# Reconstruct MultiIndex, keeping level 1 (quality codes) intact
df_summary_proportions.columns = pd.MultiIndex.from_arrays(
    [
        new_level0_columns,                              # new level 0
        df_summary_proportions.columns.get_level_values(1)          # keep level 1 (quality code)
    ]
)

# Merge columns with same name
df_summary_proportions = df_summary_proportions.T.groupby(level=[0, 1]).agg(lambda x: x.dropna().iloc[0] if x.notna().any() else pd.NA).T


In [ ]:
# For each gauge type, plot a boxplot showing the proportion of data from each gauge with a quality flag of 1. 

# Only show stats for water level data, rainfall and temperature (omit SNR)

# Create horizontal boxplots
fig, ax = plt.subplots(
    1, 1,
    figsize=(10, 3)
)

# Filter quality flag '1' columns for plotting
mask = df_summary_proportions.columns.get_level_values(1).isin(['1', 1])
ax_data_all = 100 * df_summary_proportions.loc[:, mask].fillna(0)

# Extract proportion values and label for each parameter
ax_data = []
ax_ylabels = []
for parameter in gauge_quality_proportion_summaries.keys():
    parameter_data = ax_data_all.loc[gauge_quality_proportion_summaries[parameter].keys()].filter(like=parameter)
    ax_data.append(parameter_data.values.ravel())
    ax_ylabels.append(parameter_data.columns[0][0])

# Create labels
location_counts = np.array([len(d) for d in ax_data])
stats_labels = np.array([
    f'{np.median(d):.1f}% ({np.min(d):.1f}%–{np.max(d):.1f}%)' for d in ax_data
])

# Order parameters by location count
order = np.argsort(location_counts)
ax_data = [ax_data[i] for i in order]
ax_ylabels = np.array(ax_ylabels)[order]
location_counts = location_counts[order]
stats_labels = stats_labels[order]

# Y positions of violins and boxes (1..N)
y_positions = np.arange(1, len(ax_data) + 1)

ax.boxplot(
    ax_data,
    vert=False,
    widths=0.80,
    patch_artist=True,
    # showfliers=False,
    flierprops=dict(
        marker='x',
        markersize=3,
        markeredgecolor='black',
        alpha=0.6
    ),
    boxprops=dict(facecolor='none', edgecolor='black'),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)

ax.set_xlabel('Percentage (%)')
ax.set_ylabel('Parameter')
ax.set_yticks(y_positions)
ax.set_yticklabels(ax_ylabels)
ax.set_xlim(0, 100)

# Add labels to the right of the plot
for y, count, stats_label in zip(y_positions, location_counts, stats_labels):
    label = f' {count} gauges: {stats_label}'
    ax.text(
        x=ax.get_xlim()[1],   # right edge of plot
        y=y,
        s=label,
        va='center',
        ha='left',
        fontstyle='italic'
    )


plt.tight_layout()
plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Parameter level quality flag 1 distributions.png', dpi=600, bbox_inches='tight')

plt.close(fig)


In [ ]:
# Total number of data points with each quality code

df_summary_counts.columns.unique()

gauge_quality_summary = {}
for column in df_summary_counts.columns.unique():
    # Calculate the number of data points at each location with each quality flag
    mask = df_summary_counts.columns.get_level_values(1).isin(['1', 1])
    ax_data = 100 * df_summary_counts.loc[:, mask]


    data_point_counts = df_summary_counts[column].sum()
    gauge_quality_summary[f'Quality {column} count'] = [data_point_counts]


gauge_quality_summary_counts = pd.DataFrame(gauge_quality_summary)
gauge_quality_summary_counts

In [ ]:
# Catchment level metrics: Total count and proportion of data points with each quality control label
# (as percentage of all data and as percentage of flagged data)

# Total number of data points with each quality code (consisting of flag and label)
column_counts = df_summary_counts.sum()

# Total number of data points
data_count_total = column_counts.sum()
print(f'Total number of data points: {data_count_total}\n')

# Total number of data points with quality flag = 1 (all checks passed)
data_count_passed = column_counts[column_counts.index.get_level_values(1).str.contains('1')].sum()
percent_passed = 100 * data_count_passed / data_count_total
print(f'{round(percent_passed, 1)} % of all data points have passed all quality checks')

# Total number of data points with quality flag = 2 ('suspect')
data_count_flag2 = column_counts[column_counts.index.get_level_values(1).str.contains('2')].sum()
percent_flag2 = 100 * data_count_flag2 / data_count_total
print(f'{round(percent_flag2, 1)} % of all data points have quality flag 2')

# Total number of data points with quality flag = 3 ('bad')
data_count_flag3 = column_counts[column_counts.index.get_level_values(1).str.contains('3')].sum()
percent_flag3 = 100 * data_count_flag3 / data_count_total
print(f'{round(percent_flag3, 1)} % of all data points have quality flag 3\n')


# Identify proportion of data quality codes containing each label
for label in ['A', 'B', 'C', 'D', 'E']:
    
    # Count data points containing label
    data_count_label = column_counts[column_counts.index.get_level_values(1).str.contains(label)].sum()
    
    # Calculate as percentage of all data points
    percent_of_total = 100 * data_count_label / data_count_total
    
    # Calculate as percent of flagged data (flag != 1)
    percent_of_flagged = 100 * data_count_label / (data_count_total - data_count_passed)
    
    print(f'{round(percent_of_total, 1)} % of all data points have a {label} quality label, ',
          f'{round(percent_of_flagged, 1)} % of flagged data points have a {label} quality label')



In [ ]:
# Gauge-level metrics: Median oercentage of data points with each flag

# Total data points at each gauge
gauge_total_data_points = df_summary_counts.sum(axis=1)

# Calculate number and proportion of data points with each flag at each gauge (number and proportion)
for flag in [QCFlags.Good, QCFlags.Suspect, QCFlags.Bad]:
    print(f'Quality flag {flag.value} ({flag.name})')
    columns = df_summary_counts.columns[
        df_summary_counts.columns.get_level_values(1).str.contains(str(flag.value))
        ]
    gauge_counts = df_summary_counts[columns].sum(axis=1)
    gauge_proportions = gauge_counts / gauge_total_data_points
    print(f'    Gauge-level median percentage of data points with flag {flag.value}: {round(100 * gauge_proportions.median(), 2)}')

